# Exploring IOPs. 

This notebook will create a dataframe from the meta data of each file. I will then be able to explore spatial and temporal overlaps. 


In [2]:
import pandas as pd
import numpy as np
import math
import datetime as dt
from matplotlib import pyplot as plt

import os
import shutil
import glob

from netCDF4 import Dataset
import xarray as xr

from scipy.interpolate import griddata
from scipy.signal import argrelextrema, find_peaks

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from config import load_config


In [3]:
# test loading config 
paths = load_config('paths.yml')
usc_dir =  paths['paths']['usc_interim_dir']
uiuc_dir =  paths['paths']['uiuc_interim_dir']
lidar_dir = paths['paths']['lidar_dir']

print(lidar_dir)
print(uiuc_dir)
print(usc_dir)

C:/Users/ben/OneDrive - University of South Carolina/SAVANT_Data/01_interim
C:/Users/ben/OneDrive - University of South Carolina/SAVANT_Data/01_interim/UIUC
C:/Users/ben/OneDrive - University of South Carolina/SAVANT_Data/01_interim/USC


In [4]:
os.listdir(uiuc_dir)

['09232018',
 '09292018',
 '10152018',
 '10182018',
 '10232018',
 '10272018',
 '10292018',
 '11022018',
 '11072018',
 '11102018',
 '11112018',
 '11142018']

In [5]:
print('USC Data')
for iop_folder in os.listdir(usc_dir):
    date = str(iop_folder)
    year = date[-4:]
    day = date[2:4]
    month = date[0:2]
    c = len(os.listdir(os.path.join(usc_dir ,iop_folder)))
    
    print(f'{month} {day}: {c} files')

print('UIUC Data')
for iop_folder in os.listdir(uiuc_dir):
    date = str(iop_folder)
    year = date[-4:]
    day = date[2:4]
    month = date[0:2]
    c = len(os.listdir(os.path.join(uiuc_dir ,iop_folder)))
    
    print(f'{month} {day}: {c} files') 
    

USC Data
10 27: 820 files
11 02: 10006 files
11 07: 203 files
11 10: 2327 files
11 11: 4876 files
11 13: 703 files
UIUC Data
09 23: 822 files
09 29: 669 files
10 15: 4706 files
10 18: 4474 files
10 23: 5598 files
10 27: 2562 files
10 29: 3982 files
11 02: 10035 files
11 07: 1550 files
11 10: 2246 files
11 11: 4543 files
11 14: 4862 files


Pulling the code below from previous research, need to update it. 

In [ ]:
uiuc = pd.DataFrame()
uiuc_meta = pd.DataFrame(columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])

for file in uiuc_files:
    f = os.path.join(uiuc_directory, file)
    meta_data = []
    shot_data = []
    
    bin_width = 3.5
    lidar_height = 2
    origin_X = 293946.1   #293946.1075
    origin_Y = 393318.1   #393318.1287
    origin_Z = 233.6      #233.589
    origin_Azimuth = 296
    origin_Zenith = 0
    
    with open(f, 'r') as file:
        content = file.readlines()
        meta = content[0:7]
        header1 = meta[0]
        header2 = meta[1]
        header3 = meta[2]
        header4 = meta[3]
        header5 = meta[4]
        header6 = meta[5]
        header7 = meta[6]
        
        header2 = header2.split(' ')
        campaign = header2[0]
        startDate = f'{header2[1]} {header2[2]}'
        endDate = f'{header2[3]} {header2[4]}'
        elevation = float(header2[5])
        longitude = float(header2[6])
        latitude = float(header2[7])
        zenith = float(header2[8])*-1
        azimuth = float(header2[9])
        temp_ground = float(header2[10])
        pressure = float(header2[11])
        bin_width = float(header5.split()[6])
    
    df = pd.read_csv(f, skiprows=14, nrows=500)
    df['source'] = 'UIUC'
    df['timestamp'] = pd.to_datetime(startDate, dayfirst=True)
    df['distance'] = (df.index+1)*bin_width 
    df['dist_lidar'] = df['distance'] * np.cos(np.radians(zenith)) ### !!! WORKING HERE NEEDS TO BE CORRECTED FOR DISTANCE FROM LIDAR.  
    df['azimuth'] = azimuth
    df['zenith'] = zenith
    df['analog'] = df['0 (mV)']
    df['true_azimuth'] = azimuth + origin_Azimuth
    df['true_zenith'] = zenith + origin_Zenith
    
    
    df['x'] = origin_X + df['distance'] * np.cos(np.radians(zenith+origin_Zenith)) * np.sin(np.radians(azimuth+origin_Azimuth))
    df['y'] = origin_Y + df['distance'] * np.cos(np.radians(zenith+origin_Zenith)) * np.cos(np.radians(azimuth+origin_Azimuth))
    df['z'] = origin_Z + df['distance'] * np.sin(np.radians(zenith+origin_Zenith)) + lidar_height

    uiuc = pd.concat([uiuc, df], axis=0, ignore_index=True)

    meta_data.append([f, startDate, endDate, elevation, longitude, latitude, zenith, azimuth, temp_ground, pressure])
    
    # Create DF
    temp_meta_df = pd.DataFrame(meta_data, columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])
    
    # Convert date columns to datetime
    temp_meta_df['start'] = pd.to_datetime(temp_meta_df['start'], errors='coerce', dayfirst=True)
    temp_meta_df['end'] = pd.to_datetime(temp_meta_df['end'], errors='coerce', dayfirst=True)
    uiuc_meta = pd.concat([uiuc_meta, temp_meta_df], ignore_index=True)

uiuc_meta.head()

In [ ]:
usc = pd.DataFrame()
usc_meta = pd.DataFrame(columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])

for file in usc_files:
    f = os.path.join(usc_directory, file)
    meta_data = []
    shot_data = []
    
    bin_width = 7.5
    lidar_height = 2
    origin_X = 293677.5 #293701.1 # 293705.8   #293705.7868 # 293677.5, 393553.7
    origin_Y = 393553.7 #393557.9 #393522.7   #393522.7423
    origin_Z = 337 #236.8 #235.5      #235.4768
    origin_Azimuth = 118 #120 #110
    origin_Zenith = 0
    
    with open(f, 'r') as file:
        content = file.readlines()
    
        # The first seven lines are meta data
        meta = content[0:7]
        
        header1 = meta[0]
        header2 = meta[1]
        header3 = meta[2]
        header4 = meta[3]
        header5 = meta[4]
        header6 = meta[5]
        header7 = meta[6]
        
        header2 = header2.split(' ')
        campaign = header2[0]
        startDate = f'{header2[3]} {header2[4]}'
        endDate = f'{header2[5]} {header2[6]}'
        elevation = float(header2[7])
        longitude = float(header2[8])
        latitude = float(header2[9])
        zenith = float(header2[10])*-1
        azimuth = float(header2[11])
        temp_ground = float(header2[12])
        pressure = float(header2[13])
        bin_width = float(header5.split()[6])
        data = content[7:7+200]
    
    my_data = []
    for row in data:
        d = row.strip().split('\t')
        my_data.append(d)
    df = pd.DataFrame(my_data, columns=['analog', 'photon'])
    df = df.apply(pd.to_numeric)
    df['angle'] = azimuth
    df['start'] = startDate
    df['start'] = pd.to_datetime(df['start'], dayfirst=True)
    df['end'] = endDate
    df['end'] = pd.to_datetime(df['end'], dayfirst=True)
    
    
    df['source'] = 'USC'
    df['timestamp'] = df['start']
    df['distance'] = (df.index+1)*bin_width
    df['dist_lidar'] = df['distance'] * np.cos(np.radians(zenith))
    df['azimuth'] = azimuth
    df['zenith'] = zenith
    # df['analog'] = df['0 (mV)']
    df['true_azimuth'] = azimuth+origin_Azimuth
    
    df['x'] = origin_X + df['distance'] * np.cos(np.radians(zenith+origin_Zenith)) * np.sin(np.radians(azimuth+origin_Azimuth))
    df['y'] = origin_Y + df['distance'] * np.cos(np.radians(zenith+origin_Zenith)) * np.cos(np.radians(azimuth+origin_Azimuth))
    df['z'] = origin_Z + df['distance'] * np.sin(np.radians(zenith+origin_Zenith)) + lidar_height
    df
    usc = pd.concat([usc, df], axis=0, ignore_index=True)

    meta_data.append([f, startDate, endDate, elevation, longitude, latitude, zenith, azimuth, temp_ground, pressure])
    
    # Create DF
    temp_meta_df = pd.DataFrame(meta_data, columns=['file', 'start', 'end', 'elevation', 'longitude', 'latitude', 'zenith', 'azimuth', 'temp_ground', 'pressure'])
    
    # Convert date columns to datetime
    temp_meta_df['start'] = pd.to_datetime(temp_meta_df['start'], errors='coerce', dayfirst=True)
    temp_meta_df['end'] = pd.to_datetime(temp_meta_df['end'], errors='coerce', dayfirst=True)
    usc_meta = pd.concat([usc_meta, temp_meta_df], ignore_index=True)




In [ ]:
# time correction for November 2
# usc_data['timestamp']=usc_data['timestamp'] + pd.Timedelta(hours=7) 